# Music2Vec Embedding Extractor

In [ ]:
!pip install muq yt-dlp

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
# Google Colab
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os
import hashlib
import tempfile
import torch
import soundfile as sf
import numpy as np
from muq import MuQMuLan
from tqdm.notebook import tqdm
import subprocess
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# # Google Colab
# RAW_AUDIO_DIR = '/content/drive/MyDrive/music2vec/songs'
# CACHE_DIR = '/content/drive/MyDrive/music2vec/embeddings_cache'

# On-device
YT_IDS_FILE = 'yt_ids.txt'
CACHE_DIR = 'yt_embeddings_cache'

os.makedirs(CACHE_DIR, exist_ok=True)

Using device: cuda


In [2]:
print("Loading MuLan model...")
mulan = MuQMuLan.from_pretrained("OpenMuQ/MuQ-MuLan-large")
mulan = mulan.to(device).eval()
model_lock = threading.Lock()
print("Model loaded.")

Loading MuLan model...


c:\Users\matth\AppData\Local\Python\pythoncore-3.10-64\lib\site-packages\torch\nn\utils\weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded.


In [ ]:
import tempfile
import subprocess
import torchaudio
import os
import torch
import json
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import yt_dlp

# Configuration: choose 'file' or 'playlist'
SOURCE_TYPE = "playlist" # Set to 'playlist' to use the URL below
PLAYLIST_URL = "https://www.youtube.com/playlist?list=PLB02wINShjkBKnLfufaEPnCupGO-SK6e4"

yt_ids = []

if SOURCE_TYPE == "file":
    if not os.path.exists(YT_IDS_FILE):
        print(f"Cannot find {YT_IDS_FILE}")
    else:
        with open(YT_IDS_FILE, 'r') as f:
            yt_ids = [line.strip() for line in f if line.strip()]
elif SOURCE_TYPE == "playlist":
    print(f"Extracting IDs from playlist: {PLAYLIST_URL}")
    ydl_opts_extract = {
        'extract_flat': True, 
        'quiet': True,
        'ignoreerrors': True,
        'sleep_interval': 10,
        'max_sleep_interval': 30,
        'sleep_requests': 1
    }
    with yt_dlp.YoutubeDL(ydl_opts_extract) as ydl:
        info = ydl.extract_info(PLAYLIST_URL, download=False)
        if 'entries' in info:
            yt_ids = [entry['id'] for entry in info['entries'] if entry]
        else:
            yt_ids = [info['id']]

print(f"Found {len(yt_ids)} YouTube IDs.")

def process_yt_id(yt_id):
    cache_path = os.path.join(CACHE_DIR, f"{yt_id}.pt")
    if os.path.exists(cache_path):
        return
        
    temp_dir = tempfile.mkdtemp()
    audio_path = os.path.join(temp_dir, f"{yt_id}.mp3")
    ydl_opts = {
        'format': 'bestaudio/best',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '128',
        }],
        'outtmpl': os.path.join(temp_dir, '%(id)s.%(ext)s'),
        'quiet': True,
        'no_warnings': True,
        'postprocessor_args': [
            '-ar', '24000', '-ac', '1'
        ],
        'sleep_interval': 10,
        'max_sleep_interval': 30,
        'sleep_requests': 1,
        'ratelimit': 5000000,
        'ignoreerrors': True,
        'download_archive': 'archive.txt'
    }
    try:
        # Download
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            # We use extract_info with download=True so we get metadata returned from YoutubeDL
            info = ydl.extract_info(f'https://www.youtube.com/watch?v={yt_id}', download=True)
            
            if info:
                metadata = {
                    'title': info.get('title', 'Unknown Title'),
                    'artist': info.get('uploader', 'Unknown Artist')
                }
                meta_path = os.path.join(CACHE_DIR, f"{yt_id}.json")
                with open(meta_path, 'w', encoding='utf-8') as f:
                    json.dump(metadata, f, ensure_ascii=False)
        
        if not os.path.exists(audio_path):
            return

        # Directly read the downloaded file without running ffmpeg again
        wav_tensor, sr = torchaudio.load(audio_path)
        if wav_tensor.shape[0] > 1:
            wav_tensor = wav_tensor.mean(dim=0, keepdim=True)
            
        max_val = torch.max(torch.abs(wav_tensor))
        if max_val > 0:
            wav_tensor = wav_tensor / max_val
            
        wav_tensor = wav_tensor.to(device)
        
        with torch.no_grad():
            with model_lock: # Ensure thread-safe model inference
                embed = mulan(wavs=wav_tensor)
            
        torch.save(embed[0].unsqueeze(0).cpu(), cache_path)
        
    except Exception as e:
        print(f"Error handling {yt_id}: {e}")
    finally:
        # We only need to clean up audio_path and temp_dir now
        if os.path.exists(audio_path):
            try: os.remove(audio_path)
            except: pass
        if os.path.exists(temp_dir):
            try:
                for file in os.listdir(temp_dir):
                    os.remove(os.path.join(temp_dir, file))
                os.rmdir(temp_dir)
            except: pass

print("Processing files concurrently...")
max_workers = 4 

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = [executor.submit(process_yt_id, yid) for yid in yt_ids]
    for future in tqdm(as_completed(futures), total=len(yt_ids), desc="Processing Pipeline"):
        try:
            future.result()
        except Exception as e:
            print(f"File processing failed: {e}")

print("Done!")

Extracting IDs from playlist: https://www.youtube.com/playlist?list=PLB8UhGAVn2kBEvlkzog8wiUxQk30ad_Yp
Found 20 YouTube IDs.
Processing files concurrently...


Processing Pipeline:   0%|          | 0/20 [00:00<?, ?it/s]

Done!                                                      
